# Grid2D Figures

This notebook plots a single 2D perturbation grid generated by `run_grid2d.py`. It reads the full dataset when available and falls back to the smoke dataset.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("data/grid2d")
plt.rcParams.update({"figure.dpi": 140})

def load_config(dataset_dir):
    with open(dataset_dir / "config.json") as f:
        return json.load(f)

def choose_dataset(data_dir, experiment_name="grid2d"):
    candidates = []
    for dataset_dir in sorted(data_dir.iterdir()) if data_dir.exists() else []:
        if not (dataset_dir / "config.json").exists() or not (dataset_dir / "results.npz").exists():
            continue
        config = load_config(dataset_dir)
        if config.get("experiment_name") != experiment_name:
            continue
        candidates.append(("__smoke" in dataset_dir.name, dataset_dir.stat().st_mtime, dataset_dir.name, dataset_dir, config))
    if not candidates:
        raise FileNotFoundError(f"No {experiment_name} dataset found in {data_dir}. Run python run_grid2d.py first.")
    candidates.sort(key=lambda item: (item[0], -item[1]))
    _, _, name, dataset_dir, config = candidates[0]
    results = np.load(dataset_dir / "results.npz", allow_pickle=False)
    return name, config, results

In [ ]:
grid_dataset, grid_config, grid_data = choose_dataset(DATA_DIR)

print("dataset =", grid_dataset)
print("experiment =", grid_config["experiment_name"])
print("N =", grid_config["n"])
print("delta indices =", grid_config["delta_indices"])
print("QFIM =")
print(grid_data["qfim"])

## Exact Error And QFIM Contours

In [ ]:
delta_0 = grid_data["delta_0_values"]
delta_1 = grid_data["delta_1_values"]
exact = grid_data["exact_error_grid"]
qfi = grid_data["qfi_error_grid"]

levels = [0.05, 0.10, 0.20, 0.30]
levels = [level for level in levels if level < np.nanmax(exact)]

fig, ax = plt.subplots(figsize=(5.2, 4.4))
mesh = ax.pcolormesh(delta_0, delta_1, exact, shading="auto", cmap="viridis")
fig.colorbar(mesh, ax=ax, label="exact simulation error")

if levels:
    ax.contour(delta_0, delta_1, exact, levels=levels, colors="white", linewidths=1.1)
    ax.contour(delta_0, delta_1, qfi, levels=levels, colors="black", linewidths=1.2, linestyles="--")

ax.set_xlabel(r"$\delta_0$ (xfield)")
ax.set_ylabel(r"$\delta_1$ (zfield)")
ax.set_title("Exact error landscape and QFIM ellipses")
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()

## Local Approximation Ratio

In [ ]:
ratio = grid_data["ratio_grid"]
ratio_plot = np.ma.masked_invalid(ratio)
vmin, vmax = np.nanpercentile(ratio, [5, 95])
if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
    vmin, vmax = np.nanmin(ratio), np.nanmax(ratio)

cmap = plt.get_cmap("coolwarm").copy()
cmap.set_bad("lightgray")

fig, ax = plt.subplots(figsize=(5.2, 4.4))
mesh = ax.pcolormesh(delta_0, delta_1, ratio_plot, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax)
fig.colorbar(mesh, ax=ax, label=r"$E_{exact}^2 / E_{QFIM}^2$")
ax.set_xlabel(r"$\delta_0$ (xfield)")
ax.set_ylabel(r"$\delta_1$ (zfield)")
ax.set_title("Where the local QFIM approximation breaks down")
ax.set_aspect("equal", adjustable="box")
fig.tight_layout()